# White's first-move advantage on Lichess

This notebook runs the analysis in this repository on Google Colab. It streams monthly game files
from the [Lichess Open Database](https://database.lichess.org) straight into the program (nothing
big is ever saved to disk), measures White's advantage in rating points, and saves the results.

**Just press Runtime → Run all.** When Colab asks for access to Google Drive, allow it.

### Why this run is safe

| Risk | What the notebook does about it |
|---|---|
| Colab disconnects or the runtime is reset | Everything is saved to **Google Drive** as it goes, in `MyDrive/first_mover_runs/<RUN_NAME>/` |
| Crash after hours of work | Each month's counts are saved every `CHECKPOINT_EVERY` games, and as soon as that month finishes |
| A half-written file | Files are written to a `.tmp` file first and renamed only when complete |
| The download drops or stalls | The download reconnects and continues from the exact byte where it stopped |
| A cut-off or corrupt download counted as a full month | Every month is checked: all bytes arrived, the SHA256 checksum matches Lichess's, and the zstd stream ended cleanly. Only then is it marked `DONE` |
| A month fails anyway | It is retried automatically (`MAX_ATTEMPTS` times) |
| Starting again from zero | **Run all again with the same `RUN_NAME`**: finished months are skipped, only unfinished ones run |
| A bug found only after hours | A short test run (`SMOKE_TEST_GAMES`) runs first and stops the notebook if anything is wrong |
| Losing the printed output | Every program and download log is saved to Drive too, next to the results |

**What you get** in `MyDrive/first_mover_runs/<RUN_NAME>/final/` (and as a `results.zip` download):
`counts.csv`, `output.csv`, `openings.csv`, `summary.txt`, `dashboard.html`.
Put them in `output/final/` of the repository and push.

Keep the tab open while it runs. A month that was cut off half-way has to be read again from its
start (a compressed stream cannot be resumed in the middle), but finished months are never redone.

## 1. Settings

The only cell you might edit. The results in this repository use January and February 2024.
`RUN_NAME` is the folder on Drive; keep it the same to resume a run, change it to start a new one.

In [ ]:
MONTHS = ["2024-01", "2024-02"]       # YYYY-MM, see database.lichess.org
RUN_NAME = "final_2024-01_2024-02"     # folder on Drive; the same name resumes a run

CHECKPOINT_EVERY = 2_000_000           # save each month's counts every this many games
SMOKE_TEST_GAMES = 300_000             # test run before the full run (0 = skip the test)
MAX_ATTEMPTS = 3                       # tries per month before giving up
RUN_IN_PARALLEL = True                 # read all months at the same time (faster)
STATUS_EVERY_SECONDS = 60              # how often the progress table is printed
BUILD_REPORT = True                    # step 6: build finalReport.pdf (installs LaTeX, a few minutes)

USE_DRIVE = True                       # strongly recommended: results survive a disconnect
REPO_URL = "https://github.com/Sahil-M-Deo/The-Perks-of-Being-a-First-Mover.git"
BRANCH = "main"
LICHESS_URL = "https://database.lichess.org/standard/"

## 2. Connect Google Drive

Everything this notebook makes is saved here, so a disconnect loses nothing that was already saved.

In [ ]:
import os, sys, json, time, shutil, subprocess, hashlib, zipfile, datetime

IN_COLAB = "google.colab" in sys.modules
if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("USE_DRIVE needs Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = f"/content/drive/MyDrive/first_mover_runs/{RUN_NAME}"
else:
    SAVE_ROOT = os.path.abspath(f"first_mover_runs/{RUN_NAME}")
    print("WARNING: USE_DRIVE = False, results are only kept in this session. A disconnect loses them.")

WORK_ROOT = os.path.abspath(f"work/{RUN_NAME}")          # fast local disk, copied to SAVE_ROOT
os.makedirs(SAVE_ROOT, exist_ok=True)
os.makedirs(WORK_ROOT, exist_ok=True)
NOTEBOOK_LOG = os.path.join(SAVE_ROOT, "notebook_log.txt")


def say(msg=""):
    # prints, and keeps a copy of every message on Drive
    line = time.strftime("[%H:%M:%S] ") + msg if msg else ""
    print(line, flush=True)
    try:
        with open(NOTEBOOK_LOG, "a") as f:
            f.write(line + "\n")
    except OSError:
        pass


def save_copy(src, dst_dir):
    # copies src into dst_dir safely (temporary name, then rename); returns True if it worked
    try:
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, os.path.basename(src))
        shutil.copyfile(src, dst + ".tmp")
        os.replace(dst + ".tmp", dst)
        return True
    except OSError as e:
        say(f"WARNING: could not copy {os.path.basename(src)} to Drive ({e}); will retry")
        return False


say(f"=== session started {datetime.datetime.now():%Y-%m-%d %H:%M}, months {MONTHS} ===")
say(f"results are saved in {SAVE_ROOT}")
done = [m for m in MONTHS if os.path.exists(os.path.join(SAVE_ROOT, m, "DONE.json"))]
if done:
    say(f"already finished in an earlier session, will be skipped: {done}")

## 3. Install zstd and compile the program

Clones this repository into `repo/` and compiles `scripts/white_advantage.cpp`. It also checks
that the cloned code is the version with checkpoints (push your changes before running this).

In [ ]:
def sh(cmd, tries=1, cwd=None):
    # runs a shell command, retrying network steps; stops the notebook if it keeps failing
    for t in range(1, tries + 1):
        r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
        if r.returncode == 0:
            return r.stdout
        say(f"command failed (try {t}/{tries}): {cmd}\n{r.stdout[-2000:]}{r.stderr[-2000:]}")
        time.sleep(10 * t)
    raise RuntimeError(f"failed: {cmd}")


sh("apt-get -qq update", tries=3)
sh("apt-get -qq install -y libzstd-dev", tries=3)
sh(f"rm -rf repo && git clone -q --depth 1 -b {BRANCH} {REPO_URL} repo", tries=3)
sh("g++ -O2 -std=c++17 -pthread repo/scripts/white_advantage.cpp -o repo/scripts/white_advantage -lzstd")

PROGRAM = os.path.abspath("repo/scripts/white_advantage")
DOWNLOADER = os.path.abspath("repo/scripts/stream_download.py")
source = open("repo/scripts/white_advantage.cpp").read()
if "--checkpoint-every" not in source or not os.path.exists(DOWNLOADER):
    raise RuntimeError("The cloned repository is the OLD version without checkpoints. "
                       "Push the updated scripts/ folder to GitHub, then Run all again.")
commit = sh("git -C repo rev-parse --short HEAD").strip()
say(f"compiled white_advantage from commit {commit}")

## 4. Check the months on Lichess

Checks that every month exists, gets its size and its SHA256 checksum, and whether Lichess lets a
dropped download continue where it stopped. Stops here if a month doesn't exist.

In [ ]:
import requests

def file_name(month):
    return f"lichess_db_standard_rated_{month}.pgn.zst"


checksums = {}
try:
    r = requests.get(LICHESS_URL + "sha256sums.txt", timeout=60)
    r.raise_for_status()
    for line in r.text.splitlines():
        parts = line.split()
        if len(parts) == 2:
            checksums[parts[1].lstrip("*")] = parts[0].lower()
except Exception as e:
    say(f"WARNING: could not get Lichess's checksums ({e}); files are still checked for size and a clean end")

month_info = {}
for m in MONTHS:
    url = LICHESS_URL + file_name(m)
    # stream=True: if the server ignores the Range, don't download the whole file here
    for t in range(1, 6):
        try:
            r = requests.get(url, headers={"Range": "bytes=0-3"}, timeout=60, stream=True)
            break
        except requests.RequestException as e:
            say(f"{m}: could not reach Lichess (try {t}/5): {e}")
            if t == 5:
                raise
            time.sleep(15 * t)
    if r.status_code == 404:
        raise RuntimeError(f"{m}: no such file on Lichess ({url}). Check MONTHS.")
    if r.status_code not in (200, 206):
        raise RuntimeError(f"{m}: Lichess answered HTTP {r.status_code}")
    resumable = r.status_code == 206
    size = None
    if resumable and "/" in r.headers.get("Content-Range", ""):
        size = int(r.headers["Content-Range"].split("/")[-1])
    r.close()
    month_info[m] = {"url": url, "size": size, "sha256": checksums.get(file_name(m)), "resumable": resumable}
    say(f"{m}: {size / 1e9:.1f} GB" if size else f"{m}: size unknown")
    say(f"   checksum {'found' if month_info[m]['sha256'] else 'NOT found'}, "
        f"resume after a dropped connection: {'yes' if resumable else 'NO (a drop restarts the month)'}")

## 5. Helpers that run one month

Each month runs as two processes connected by a named pipe: `stream_download.py` downloads the
file into the pipe, and `white_advantage` reads the other end. Both write their logs and the
counts to a local folder, which is copied to Drive every few seconds.

In [ ]:
STARTED_RUNS = globals().get("STARTED_RUNS", [])


class MonthRun:
    # one attempt at one month: starts the download and the program, and checks the result

    def __init__(self, month, save_dir, attempt, max_games=0):
        self.month, self.save_dir, self.attempt, self.max_games = month, save_dir, attempt, max_games
        self.info = month_info[month]
        tag = "test" if max_games else "full"
        self.dir = os.path.join(WORK_ROOT, f"{tag}_{month}")
        shutil.rmtree(self.dir, ignore_errors=True)
        os.makedirs(self.dir)
        fifo_dir = os.path.join(self.dir, "pipe")
        os.makedirs(fifo_dir)
        # the pipe has the real file name, so the program knows the month and that it is .zst
        self.pipe = os.path.join(fifo_dir, file_name(month))
        os.mkfifo(self.pipe)
        self.status_file = os.path.join(self.dir, "download_status.json")
        self.prog_log = os.path.join(self.dir, "program_log.txt")
        self.dl_log = os.path.join(self.dir, "download_log.txt")

        args = [PROGRAM, self.pipe, "--checkpoint-every",
                str(min(CHECKPOINT_EVERY, max_games) if max_games else CHECKPOINT_EVERY)]
        if max_games:
            args += ["--max-games", str(max_games)]
        self.prog = subprocess.Popen(args, cwd=self.dir, stdout=open(self.prog_log, "w"),
                                     stderr=subprocess.STDOUT)
        self.dl = subprocess.Popen([sys.executable, DOWNLOADER, self.info["url"], self.pipe,
                                    self.status_file, self.info["sha256"] or ""],
                                   stdout=open(self.dl_log, "w"), stderr=subprocess.STDOUT)
        self.started = time.time()
        self.prog_exit_time = None
        self.dl_exit_time = None
        self.last_progress = (0, time.time())
        self.copied = {}
        self.result = None          # None while running, then True / False
        self.reason = ""

    def download_status(self):
        try:
            return json.load(open(self.status_file))
        except (OSError, ValueError):
            return {}

    def games_read(self):
        # the last progress line of the program, like "[12:00:10] 2024-01: 5000000 games read, ..."
        try:
            lines = [l for l in open(self.prog_log) if " games read" in l]
        except OSError:
            return 0
        if not lines:
            return 0
        try:
            return int(lines[-1].split(": ", 1)[1].split(" games read")[0])
        except (IndexError, ValueError):
            return 0

    def sync(self, force=False):
        # copies every new or changed file to Drive
        for name in os.listdir(self.dir):
            path = os.path.join(self.dir, name)
            if not os.path.isfile(path) or name.endswith(".tmp"):
                continue
            stamp = (os.path.getmtime(path), os.path.getsize(path))
            if force or self.copied.get(name) != stamp:
                if save_copy(path, self.save_dir):
                    self.copied[name] = stamp

    def poll(self):
        # call regularly; returns True once this attempt is over
        if self.result is not None:
            return True
        now = time.time()
        prog_rc, dl_rc = self.prog.poll(), self.dl.poll()
        st = self.download_status()

        # watchdogs, so nothing can hang forever
        progress = (st.get("bytes", 0), self.games_read())
        if progress != self.last_progress[0]:
            self.last_progress = (progress, now)
        if prog_rc is None and now - self.last_progress[1] > 20 * 60:
            say(f"{self.month}: no progress for 20 minutes, stopping this attempt")
            self.prog.kill()
            self.dl.kill()
        if prog_rc is not None:
            self.prog_exit_time = self.prog_exit_time or now
            # the program stopped (finished, test run, or crashed): the download has no reader left
            if dl_rc is None and now - self.prog_exit_time > 30:
                self.dl.kill()
        if dl_rc is not None:
            self.dl_exit_time = self.dl_exit_time or now
            # the download ended; the program reads what's left in seconds, so 5 minutes means stuck
            if prog_rc is None and now - self.dl_exit_time > 300:
                say(f"{self.month}: the program did not finish after the download ended, stopping it")
                self.prog.kill()
        if prog_rc is None or self.dl.poll() is None:
            return False

        self.check(prog_rc, self.download_status())
        self.sync(force=True)
        if self.result and not self.max_games:
            self.mark_done()
        return True

    def check(self, prog_rc, st):
        counts = os.path.join(self.dir, f"counts_{self.month}.csv")
        problems = []
        status_line, seen, has_end = "", 0, False
        if os.path.exists(counts):
            for line in open(counts):
                if line.startswith("#status,"):
                    status_line = line.strip().split(",", 1)[1]
                elif line.startswith("#month,"):
                    label, seen = line.split(",")[1], int(line.split(",")[2])
                    if label != self.month:
                        problems.append(f"the program labelled the month '{label}'")
                elif line.startswith("#end"):
                    has_end = True
        else:
            problems.append("no counts file")
        if not has_end:
            problems.append("counts file not complete")

        if self.max_games:
            if prog_rc != 0:
                problems.append(f"program exit code {prog_rc}")
            if seen != self.max_games:
                problems.append(f"read {seen:,} games, expected {self.max_games:,}")
            if st.get("state") == "failed":
                if seen == self.max_games:
                    say(f"{self.month}: note: the test's download ended with: {st.get('error')}")
                else:
                    problems.append(f"download failed: {st.get('error')}")
        else:
            if prog_rc != 0:
                problems.append(f"program exit code {prog_rc} (see program_log.txt)")
            if status_line != "complete":
                problems.append(f"counts status '{status_line}'")
            if st.get("state") != "done":
                problems.append(f"download {st.get('state')}: {st.get('error')}")
            if self.info["size"] and st.get("bytes") != self.info["size"]:
                problems.append(f"downloaded {st.get('bytes')} of {self.info['size']} bytes")

        self.seen = seen
        self.result = not problems
        self.reason = "; ".join(problems)

    def mark_done(self):
        # only after the counts are safely on Drive (read back and checked) is the month DONE
        st = self.download_status()
        saved = os.path.join(self.save_dir, f"counts_{self.month}.csv")
        try:
            text = open(saved).read()
        except OSError:
            text = ""
        if "#status,complete\n" not in text or not text.endswith("#end\n"):
            self.result = False
            self.reason = "the counts file on Drive could not be checked"
            return
        done = {"month": self.month, "games_read": self.seen, "bytes": st.get("bytes"),
                "sha256": st.get("sha256"), "sha256_checked": bool(self.info["sha256"]),
                "minutes": round((time.time() - self.started) / 60, 1), "commit": commit,
                "finished": datetime.datetime.now().isoformat(timespec="seconds")}
        path = os.path.join(self.dir, "DONE.json")
        with open(path, "w") as f:
            json.dump(done, f, indent=1)
        if not save_copy(path, self.save_dir):
            self.result = False
            self.reason = "could not save DONE.json to Drive"
            return
        try:
            os.remove(os.path.join(self.save_dir, f"partial_counts_{self.month}.csv"))
        except OSError:
            pass

    def stop(self):
        for p in (self.prog, self.dl):
            if p.poll() is None:
                p.kill()

    def status_line(self):
        st = self.download_status()
        got, total = st.get("bytes", 0), st.get("total") or self.info["size"]
        speed = st.get("mb_per_s", 0) or 0
        text = f"{self.month}  try {self.attempt}  {got / 1e9:6.2f}"
        if total:
            text += f" / {total / 1e9:.2f} GB ({100 * got / total:5.1f}%)"
            if speed > 0:
                text += f"  ETA {(total - got) / 1e6 / speed / 60:4.0f} min"
        text += f"  {speed:5.1f} MB/s  {self.games_read():,} games  {st.get('state', 'starting')}"
        if st.get("reconnects"):
            text += f"  reconnects {st['reconnects']}"
        return text


def run_months(months, save_root, max_games=0, attempts=MAX_ATTEMPTS):
    # runs every month (in parallel or one by one) until it is done or out of attempts
    # stop anything still running from an earlier run of this cell in the same session
    for old in STARTED_RUNS:
        old.stop()
    pending = {m: 0 for m in months}
    finished, failed = [], {}
    running = []
    last_status = 0.0
    try:
        while pending or running:
            while pending and (RUN_IN_PARALLEL or not running):
                m = next(iter(pending))
                pending[m] += 1
                attempt = pending.pop(m)
                say(f"{m}: starting attempt {attempt}" + (f" (test, {max_games:,} games)" if max_games else ""))
                run = MonthRun(m, os.path.join(save_root, m), attempt, max_games)
                STARTED_RUNS.append(run)
                run.tries = attempt
                running.append(run)

            time.sleep(5)
            for run in list(running):
                if not run.poll():
                    continue
                running.remove(run)
                if run.result:
                    say(f"{run.month}: OK, {run.seen:,} games read ({(time.time() - run.started) / 60:.0f} min)")
                    finished.append(run.month)
                else:
                    say(f"{run.month}: attempt {run.tries} FAILED: {run.reason}")
                    # keep the failed attempt's logs for later
                    keep = os.path.join(save_root, run.month, f"failed_attempt_{run.tries}")
                    for name in ("program_log.txt", "download_log.txt", "download_status.json",
                                 f"counts_{run.month}.csv"):
                        p = os.path.join(run.dir, name)
                        if os.path.exists(p):
                            save_copy(p, keep)
                    # a failed month's counts must not be mistaken for good ones later
                    for name in (f"counts_{run.month}.csv", "counts.csv"):
                        try:
                            os.remove(os.path.join(save_root, run.month, name))
                        except OSError:
                            pass
                    if run.tries < attempts:
                        pending[run.month] = run.tries
                    else:
                        failed[run.month] = run.reason

            for run in running:
                run.sync()
            if running and time.time() - last_status >= STATUS_EVERY_SECONDS:
                for run in running:
                    say(run.status_line())
                last_status = time.time()
    except KeyboardInterrupt:
        say("stopped by hand; saving what's there")
        for run in running:
            run.stop()
            run.sync(force=True)
        raise
    return finished, failed

## 6. Test run

Reads the first `SMOKE_TEST_GAMES` games of the first month (a minute or two) and checks the
whole chain: download, pipe, program, checkpoint files, Drive. If anything is wrong the notebook
stops here, before the long run. Skipped when every month is already done.

In [ ]:
todo = [m for m in MONTHS if not os.path.exists(os.path.join(SAVE_ROOT, m, "DONE.json"))]
if not todo:
    say("every month is already done, skipping the test run")
elif SMOKE_TEST_GAMES <= 0:
    say("SMOKE_TEST_GAMES = 0, skipping the test run")
else:
    test_root = os.path.join(SAVE_ROOT, "test_run")
    ok, bad = run_months([todo[0]], test_root, max_games=SMOKE_TEST_GAMES, attempts=2)
    if bad:
        raise RuntimeError(f"The test run failed: {bad}. Logs are in {test_root}. Nothing big was started.")

    # every opening in the report should have some games
    counts_file = os.path.join(test_root, todo[0], f"counts_{todo[0]}.csv")
    games_by_opening = {}
    for line in open(counts_file):
        if line.startswith("#") or line.startswith("opening,"):
            continue
        p = line.rstrip("\n").split(",")
        games_by_opening[p[0]] = games_by_opening.get(p[0], 0) + int(p[4])
    say("test run games per opening: " + ", ".join(f"{k} {v:,}" for k, v in sorted(games_by_opening.items())))
    import re
    names = re.findall(r'\{"([^"]+)",\s*(?:AGGRESSIVE|DRAWISH|EXTRA)', open("repo/scripts/src/openings.h").read())
    missing = [n for n in names if games_by_opening.get(n, 0) == 0]
    if missing:
        say(f"WARNING: no games for {missing} in the test run; check their names in openings.h "
            "(a rare opening can also just be missing from a small test)")
    if not os.path.exists(os.path.join(test_root, todo[0], f"counts_{todo[0]}.csv")):
        raise RuntimeError("the test run's counts were not saved to Drive")
    say("TEST RUN PASSED")

## 7. The full run

Runs every month that isn't done yet, printing a progress line per month every
`STATUS_EVERY_SECONDS`. Each month's counts are saved to Drive as it goes, and a month is marked
`DONE` only after every check passed. If the session dies, reconnect and **Run all** again.

In [ ]:
todo = [m for m in MONTHS if not os.path.exists(os.path.join(SAVE_ROOT, m, "DONE.json"))]
for m in MONTHS:
    if m not in todo:
        d = json.load(open(os.path.join(SAVE_ROOT, m, "DONE.json")))
        say(f"{m}: already done ({d['games_read']:,} games), skipping")

if todo:
    say(f"full run of {todo}, results saved to {SAVE_ROOT}")
    ok, bad = run_months(todo, SAVE_ROOT)
    if bad:
        raise RuntimeError(f"These months did not finish: {bad}. Everything else is saved. "
                           "Run all again (same RUN_NAME) to retry only these months.")

for m in MONTHS:
    if not os.path.exists(os.path.join(SAVE_ROOT, m, "DONE.json")):
        raise RuntimeError(f"{m} is not done")
say("ALL MONTHS DONE")

## 8. Combine the months and save the results

Adds the months' counts together and runs the analysis on them (seconds). Saves `counts.csv`,
`output.csv`, `openings.csv`, `summary.txt`, `dashboard.html` and `results.zip` to
`<RUN_NAME>/final/` on Drive, copies them into `repo/output/final/`, and downloads `results.zip`.

In [ ]:
final_dir = os.path.join(WORK_ROOT, "final")
shutil.rmtree(final_dir, ignore_errors=True)
os.makedirs(final_dir)
args = [PROGRAM]
for m in MONTHS:
    args += ["--from-counts", os.path.join(SAVE_ROOT, m, f"counts_{m}.csv")]
r = subprocess.run(args, cwd=final_dir, capture_output=True, text=True)
open(os.path.join(final_dir, "summary.txt"), "w").write(r.stdout)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError(f"combining the months failed (exit code {r.returncode}):\n{r.stderr}")

RESULT_FILES = ["counts.csv", "output.csv", "openings.csv", "summary.txt", "dashboard.html"]
for name in RESULT_FILES:
    if not os.path.exists(os.path.join(final_dir, name)):
        raise RuntimeError(f"{name} was not made")
zip_path = os.path.join(final_dir, "results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in RESULT_FILES:
        z.write(os.path.join(final_dir, name), name)

save_dir = os.path.join(SAVE_ROOT, "final")
for name in RESULT_FILES + ["results.zip"]:
    if not save_copy(os.path.join(final_dir, name), save_dir):
        raise RuntimeError(f"could not save {name} to {save_dir}")
os.makedirs("repo/output/final", exist_ok=True)
for name in RESULT_FILES:
    shutil.copyfile(os.path.join(final_dir, name), os.path.join("repo/output/final", name))
say(f"RESULTS SAVED in {save_dir}")

if IN_COLAB:
    try:
        from google.colab import files as colab_files
        colab_files.download(zip_path)
    except Exception as e:
        say(f"the browser download didn't start ({e}); results.zip is on Drive in {save_dir}")

## 9. (Optional) Build the report

Rebuilds `reports/final/report_numbers.tex` and `finalReport.pdf` from the combined `counts.csv`,
saves them to `<RUN_NAME>/final/` on Drive and downloads `report.zip`. The results are already
safe at this point, so a problem here loses nothing. Set `BUILD_REPORT = False` to skip it.

In [ ]:
if not BUILD_REPORT:
    say("BUILD_REPORT = False, skipping the report")
else:
    sh("apt-get -qq install -y texlive-latex-recommended texlive-latex-extra "
       "texlive-fonts-recommended texlive-pictures", tries=2)
    r = subprocess.run(["../../scripts/white_advantage", "--from-counts", "counts.csv",
                        "--report", "../../reports/final/report_numbers.tex"],
                       cwd="repo/output/final", capture_output=True, text=True)
    print(r.stdout[-3000:])
    report_files = ["reports/final/finalReport.pdf", "reports/final/report_numbers.tex"]
    made = [f for f in report_files if os.path.exists(os.path.join("repo", f))]
    for f in made:
        save_copy(os.path.join("repo", f), os.path.join(SAVE_ROOT, "final"))
    if "reports/final/finalReport.pdf" not in made:
        say("WARNING: the PDF was not built; see repo/reports/final/finalReport.log")
    else:
        report_zip = os.path.join(WORK_ROOT, "report.zip")
        with zipfile.ZipFile(report_zip, "w", zipfile.ZIP_DEFLATED) as z:
            for f in made + ["output/final/dashboard.html", "output/final/output.csv", "output/final/openings.csv"]:
                z.write(os.path.join("repo", f), os.path.basename(f))
        save_copy(report_zip, os.path.join(SAVE_ROOT, "final"))
        say(f"REPORT SAVED in {os.path.join(SAVE_ROOT, 'final')}")
        if IN_COLAB:
            try:
                from google.colab import files as colab_files
                colab_files.download(report_zip)
            except Exception as e:
                say(f"the browser download didn't start ({e}); report.zip is on Drive")